# 🎤 Qwen3-TTS — Voice Cloning & Text-to-Speech

**Qwen3-TTS** de Alibaba/Tongyi — modelo TTS con clonación de voz zero-shot.

**Requisitos:** GPU T4 (gratis en Colab) — Ve a `Runtime > Change runtime type > T4 GPU`

---

## 1. Instalación (~3-5 min)

In [ ]:
# Instalar dependencias
!pip install -q transformers accelerate soundfile torch torchaudio
!pip install -q gradio  # Para la interfaz web

# Verificar GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - activa T4 en Runtime!'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 2. Cargar modelo (~2-3 min primera vez)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import soundfile as sf
import numpy as np

MODEL_ID = "Qwen/Qwen3-TTS"

print("Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Cargando modelo...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Modelo cargado!")

## 3. Generar voz (sin clonación)

Genera audio con la voz por defecto del modelo.

In [ ]:
from IPython.display import Audio, display

# Texto a generar
texto = "Hola, soy Qwen TTS. Puedo hablar en español y clonar voces."

# Generar audio
inputs = tokenizer(texto, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.9,
    )

# Decodificar audio
audio_tokens = outputs[0][inputs['input_ids'].shape[1]:]
audio_data = tokenizer.decode_audio(audio_tokens)

# Guardar y reproducir
sf.write("output_basic.wav", audio_data, 24000)
display(Audio(audio_data, rate=24000))
print("✅ Audio guardado: output_basic.wav")

## 4. Clonación de voz (Zero-Shot)

Sube un audio de referencia (~5-15 segundos) y el modelo generará speech con esa voz.

**Sube tu audio:** Usa el botón de archivos de la izquierda (📁) o ejecuta la celda de abajo.

In [ ]:
# Subir audio de referencia
from google.colab import files
import os

print("Sube un audio de referencia (WAV, MP3, FLAC — 5-15 segundos ideal):")
uploaded = files.upload()

ref_audio_path = list(uploaded.keys())[0]
print(f"\n✅ Audio subido: {ref_audio_path}")

In [ ]:
import torchaudio
from IPython.display import Audio, display

# Cargar audio de referencia
ref_waveform, ref_sr = torchaudio.load(ref_audio_path)
if ref_sr != 24000:
    ref_waveform = torchaudio.functional.resample(ref_waveform, ref_sr, 24000)
    ref_sr = 24000

# Mono
if ref_waveform.shape[0] > 1:
    ref_waveform = ref_waveform.mean(dim=0, keepdim=True)

print(f"Audio de referencia: {ref_waveform.shape[1]/ref_sr:.1f}s")
display(Audio(ref_waveform.squeeze().numpy(), rate=ref_sr))

In [ ]:
# Texto que quieres que diga con la voz clonada
texto_clon = "Este es un ejemplo de clonación de voz con Qwen TTS. Mi voz suena como la referencia."

# Tokenizar audio de referencia
ref_audio_tokens = tokenizer.encode_audio(ref_waveform.squeeze().numpy())

# Construir prompt con referencia + texto nuevo
prompt = tokenizer.build_tts_prompt(
    text=texto_clon,
    ref_audio_tokens=ref_audio_tokens
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("Generando audio con voz clonada...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        temperature=0.7,
        top_p=0.9,
    )

# Decodificar
audio_tokens = outputs[0][inputs['input_ids'].shape[1]:]
cloned_audio = tokenizer.decode_audio(audio_tokens)

# Guardar y reproducir
sf.write("output_cloned.wav", cloned_audio, 24000)
display(Audio(cloned_audio, rate=24000))
print("✅ Audio clonado guardado: output_cloned.wav")

## 5. Interfaz web con Gradio

Lanza una interfaz web para probar cómodamente.

In [ ]:
import gradio as gr
import tempfile

def generate_tts(text, ref_audio_file, temperature, top_p):
    """Genera TTS con o sin clonación de voz."""
    try:
        if ref_audio_file is not None:
            # Con clonación
            wf, sr = torchaudio.load(ref_audio_file)
            if sr != 24000:
                wf = torchaudio.functional.resample(wf, sr, 24000)
            if wf.shape[0] > 1:
                wf = wf.mean(dim=0, keepdim=True)
            
            ref_tokens = tokenizer.encode_audio(wf.squeeze().numpy())
            prompt = tokenizer.build_tts_prompt(text=text, ref_audio_tokens=ref_tokens)
        else:
            prompt = text
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4096,
                temperature=temperature,
                top_p=top_p,
            )
        
        audio_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        audio = tokenizer.decode_audio(audio_tokens)
        
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(tmp.name, audio, 24000)
        return tmp.name
    except Exception as e:
        raise gr.Error(f"Error: {str(e)}")

with gr.Blocks(title="Qwen3-TTS — DRIFTLYA", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎤 Qwen3-TTS\n### Text-to-Speech con clonación de voz")
    
    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="Texto",
                placeholder="Escribe el texto que quieres que diga...",
                lines=3
            )
            ref_audio = gr.Audio(
                label="Audio de referencia (opcional — para clonar voz)",
                type="filepath"
            )
            with gr.Row():
                temp_slider = gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature")
                topp_slider = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-P")
            gen_btn = gr.Button("🎙️ Generar", variant="primary")
        
        with gr.Column():
            audio_output = gr.Audio(label="Audio generado", type="filepath")
    
    gen_btn.click(
        fn=generate_tts,
        inputs=[text_input, ref_audio, temp_slider, topp_slider],
        outputs=audio_output
    )
    
    gr.Markdown("---\n*Powered by Qwen3-TTS | driftlya*")

demo.launch(share=True)
print("\n🔗 Copia el enlace 'Running on public URL' de arriba para acceder desde el móvil!")

## 6. Descargar audios generados

In [ ]:
from google.colab import files
import glob

wavs = glob.glob("output_*.wav")
for w in wavs:
    print(f"Descargando {w}...")
    files.download(w)

if not wavs:
    print("No hay audios generados todavía. Ejecuta las celdas de arriba primero.")